In [ ]:
# --- THIS IS THE CONFIDENCE BANDS ONE ---

yi_name = 'zero-one-ai/Yi-34B-Chat'
yi_idx = llms.index(yi_name)

# --- Baselines (computed once) ---
y_acc_pred_xgb_baseline = xgb_multi.predict(x_test)

acc_xgb_baseline = np.zeros(len(lambdas))
cost_xgb_baseline = np.zeros(len(lambdas))
for i, lam in enumerate(lambdas):
    acc_xgb_baseline[i], cost_xgb_baseline[i] = score_auc(
      y_acc_test, y_cost_test, y_acc_pred_xgb_baseline, y_cost_test, lam
    )

y_acc_pred_leave_out = np.zeros_like(y_acc_test, dtype=float)
for j in range(y_acc_test.shape[1]):
    if j != yi_idx:
        y_acc_pred_leave_out[:, j] = y_acc_pred_xgb_baseline[:, j]

acc_leave_out = np.zeros(len(lambdas))
cost_leave_out = np.zeros(len(lambdas))
for i, lam in enumerate(lambdas):
    acc_leave_out[i], cost_leave_out[i] = score_auc(
        y_acc_test, y_cost_test, y_acc_pred_leave_out, y_cost_test, lam
    )

# --- Helper ---
def load_and_eval(directory, n_samples):
    path = f"{directory}/Yi_34B_{n_samples}.pkl"
    if not os.path.exists(path):
        return None
    with open(path, "rb") as f:
        ckpt = pickle.load(f)
    committee = ckpt["committee"] if isinstance(ckpt, dict) and "committee" in ckpt else ckpt
    preds = np.stack([m.predict(x_test) for m in committee], axis=1)
    ensemble_pred = preds.mean(axis=1)
    y_acc_pred = y_acc_pred_leave_out.copy()
    y_acc_pred[:, yi_idx] = ensemble_pred
    acc_curve = np.zeros(len(lambdas))
    cost_curve = np.zeros(len(lambdas))
    for i, lam in enumerate(lambdas):
        acc_curve[i], cost_curve[i] = score_auc(
            y_acc_test, y_cost_test, y_acc_pred, y_cost_test, lam
        )
    return acc_curve, cost_curve

# --- Directories for each seed ---
qbc_dirs = {
    42: 'qbc_xgb_important',
    43: 'qbc_XGB_checkpoints_yi_43',
    44: 'qbc_XGB_checkpoints_yi_44',
    45: 'qbc_XGB_checkpoints_yi_45',
    46: 'qbc_XGB_checkpoints_yi_46',
}

ddqbc_dirs = {
    42: 'ddqbc_XGB_checkpoints_yi',
    43: 'ddqbc_XGB_checkpoints_yi_43',
    44: 'ddqbc_XGB_checkpoints_yi_44',
    45: 'ddqbc_XGB_checkpoints_yi_45',
    46: 'ddqbc_XGB_checkpoints_yi_46',
}

sample_sizes = [1000, 1500, 2000, 2500, 3000, 3500, 4000, 4600]

# --- One plot per sample size ---
for n_samples in sample_sizes:
    fig, ax = plt.subplots(figsize=(16, 8))

    ax.plot(cost_xgb_baseline, acc_xgb_baseline, '-', color='green',
          label='XGBoost Baseline (all trained)', linewidth=2)
    ax.plot(cost_leave_out, acc_leave_out, '-', color='red',
          label='Leave Yi Out', linewidth=2)

    # --- QBC across seeds ---
    qbc_acc_all, qbc_cost_all = [], []
    for seed, d in qbc_dirs.items():
        result = load_and_eval(d, n_samples)
        if result is not None:
            qbc_acc_all.append(result[0])
            qbc_cost_all.append(result[1])

    if qbc_acc_all:
        qbc_acc = np.array(qbc_acc_all)
        qbc_cost = np.array(qbc_cost_all)
        acc_mean = qbc_acc.mean(axis=0)
        cost_mean = qbc_cost.mean(axis=0)
        ax.plot(cost_mean, acc_mean, '-', color='blue',
              label=f'QBC ({len(qbc_acc_all)} seed{"s" if len(qbc_acc_all) > 1 else ""})',
              linewidth=2)
        if len(qbc_acc_all) > 1:
            acc_std = qbc_acc.std(axis=0)
            ax.fill_between(cost_mean, acc_mean - acc_std, acc_mean + acc_std,
                          color='blue', alpha=0.15)

    # --- DD-QBC across seeds ---
    ddqbc_acc_all, ddqbc_cost_all = [], []
    for seed, d in ddqbc_dirs.items():
        result = load_and_eval(d, n_samples)
        if result is not None:
            ddqbc_acc_all.append(result[0])
            ddqbc_cost_all.append(result[1])

    if ddqbc_acc_all:
        ddqbc_acc = np.array(ddqbc_acc_all)
        ddqbc_cost = np.array(ddqbc_cost_all)
        acc_mean = ddqbc_acc.mean(axis=0)
        cost_mean = ddqbc_cost.mean(axis=0)
        ax.plot(cost_mean, acc_mean, '--', color='darkorange',
              label=f'DD-QBC ({len(ddqbc_acc_all)} seeds)',
              linewidth=2)
        if len(ddqbc_acc_all) > 1:
            acc_std = ddqbc_acc.std(axis=0)
            ax.fill_between(cost_mean, acc_mean - acc_std, acc_mean + acc_std,
                          color='darkorange', alpha=0.15)

    # --- LLM scatter ---
    markers = ['v', '^', '<', '>', '1', '2', '3', '4', '8', 's', 'p']
    colors_llms = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'brown', 'purple', 'gray', 'red']
    for j, _ in enumerate(llms):
        avg_acc = np.mean(y_acc_test[:, j])
        avg_cost = np.mean(y_cost_test[:, j]) / np.max(y_cost_test)
        ax.scatter(avg_cost, avg_acc, marker=markers[j],
                 color=colors_llms[j], s=80, alpha=0.7, label=llms_short[j])

    ax.set_ylim(0.65, 0.825)
    ax.set_xlim(-0.01, 0.25)
    ax.set_xlabel('Cost', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'DD-QBC vs QBC — {n_samples} samples', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# -- THIS IS WARM START REFERENCE --

yi_name = 'zero-one-ai/Yi-34B-Chat'
yi_idx = llms.index(yi_name)

# baseline
y_acc_pred_xgb_baseline = xgb_multi.predict(x_test)

acc_xgb_baseline = np.zeros(len(lambdas))
cost_xgb_baseline = np.zeros(len(lambdas))
for i, lam in enumerate(lambdas):
    acc_xgb_baseline[i], cost_xgb_baseline[i] = score_auc(
        y_acc_test, y_cost_test, y_acc_pred_xgb_baseline, y_cost_test, lam
    )

# leave yi out
y_acc_pred_leave_out = np.zeros_like(y_acc_test, dtype=float)
for j in range(y_acc_test.shape[1]):
    if j != yi_idx:
        y_acc_pred_leave_out[:, j] = y_acc_pred_xgb_baseline[:, j]

acc_leave_out = np.zeros(len(lambdas))
cost_leave_out = np.zeros(len(lambdas))
for i, lam in enumerate(lambdas):
    acc_leave_out[i], cost_leave_out[i] = score_auc(
        y_acc_test, y_cost_test, y_acc_pred_leave_out, y_cost_test, lam
    )

def load_and_eval(directory, n_samples):
    path = f"{directory}/Yi_34B_{n_samples}.pkl"
    if not os.path.exists(path):
        print(f"[missing] {path}")
        return None
    with open(path, "rb") as f:
        ckpt = pickle.load(f)
    committee = ckpt["committee"] if isinstance(ckpt, dict) and "committee" in ckpt else ckpt
    preds = np.stack([m.predict(x_test) for m in committee], axis=1)
    ensemble_pred = preds.mean(axis=1)
    y_acc_pred = y_acc_pred_leave_out.copy()
    y_acc_pred[:, yi_idx] = ensemble_pred
    acc_curve = np.zeros(len(lambdas))
    cost_curve = np.zeros(len(lambdas))
    for i, lam in enumerate(lambdas):
        acc_curve[i], cost_curve[i] = score_auc(
            y_acc_test, y_cost_test, y_acc_pred, y_cost_test, lam
        )
    return acc_curve, cost_curve

directories = [
    "warmstart100_ddqbc_XGB_checkpoints_yi_42",
    "warmstart150_ddqbc_XGB_checkpoints_yi_42",
    "warmstart200_ddqbc_XGB_checkpoints_yi_42",
    "warmstart250_ddqbc_XGB_checkpoints_yi_42",
    "warmstart300_ddqbc_XGB_checkpoints_yi_42",
    "warmstart350_ddqbc_XGB_checkpoints_yi_42",
    "warmstart400_ddqbc_XGB_checkpoints_yi_42",
    "warmstart450_ddqbc_XGB_checkpoints_yi_42",
    "warmstart500_ddqbc_XGB_checkpoints_yi_42",
]

# --- Warm-start DD-QBC ---
for d in directories:
    ws_sizes = [100, 500, 1000, 1500, 2000]
    ws_curves = {}
    for n in ws_sizes:
        result = load_and_eval(d, n)
        if result is not None:
            ws_curves[n] = result

    # --- Plot ---
    plt.figure(figsize=(20, 8))
    plt.ylim(0.65, 0.825)
    plt.xlim(-0.01, 0.25)

    plt.plot(cost_xgb_baseline, acc_xgb_baseline, '-', color='green',
             label='XGBoost Baseline (all trained)', linewidth=2)
    plt.plot(cost_leave_out, acc_leave_out, '-', color='red',
             label='Leave Yi Out', linewidth=2)

    ws_colors = ['navy', 'darkorange', 'dodgerblue', 'slateblue', "r"]
    for i, n in enumerate(ws_sizes):
        if n in ws_curves:
            acc_curve, cost_curve = ws_curves[n]
            plt.plot(cost_curve, acc_curve, '--',
                     color=ws_colors[i],
                     label=f'WS-DDQBC {n} samples',
                     linewidth=1.5)

    markers = ['v', '^', '<', '>', '1', '2', '3', '4', '8', 's', 'p']
    colors_llms = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'brown', 'purple', 'gray', 'red']
    for j, _ in enumerate(llms):
        avg_acc = np.mean(y_acc_test[:, j])
        avg_cost = np.mean(y_cost_test[:, j]) / np.max(y_cost_test)
        plt.scatter(avg_cost, avg_acc, marker=markers[j],
                    color=colors_llms[j], s=80, alpha=0.7, label=llms_short[j])

    plt.xlabel('Cost', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title(f'Warm Start DC QBC Active Learning Performance(warm start size: {d[9:12]}', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# -- This is QBC Reference --
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

yi_name = 'zero-one-ai/Yi-34B-Chat'
yi_idx = llms.index(yi_name)

# baseline
y_acc_pred_xgb_baseline = xgb_multi.predict(x_test)

acc_xgb_baseline = np.zeros(len(lambdas))
cost_xgb_baseline = np.zeros(len(lambdas))
for i, lam in enumerate(lambdas):
    acc_xgb_baseline[i], cost_xgb_baseline[i] = score_auc(
        y_acc_test, y_cost_test, y_acc_pred_xgb_baseline, y_cost_test, lam
    )

# leave yi out
y_acc_pred_leave_out = np.zeros_like(y_acc_test, dtype=float)
for j in range(y_acc_test.shape[1]):
    if j != yi_idx:
        y_acc_pred_leave_out[:, j] = y_acc_pred_xgb_baseline[:, j]

acc_leave_out = np.zeros(len(lambdas))
cost_leave_out = np.zeros(len(lambdas))
for i, lam in enumerate(lambdas):
    acc_leave_out[i], cost_leave_out[i] = score_auc(
        y_acc_test, y_cost_test, y_acc_pred_leave_out, y_cost_test, lam
    )

def load_and_eval(directory, n_samples):
    path = f"{directory}/Yi_34B_{n_samples}.pkl"
    if not os.path.exists(path):
        print(f"[missing] {path}")
        return None

    with open(path, "rb") as f:
        ckpt = pickle.load(f)

    committee = ckpt["committee"] if isinstance(ckpt, dict) and "committee" in ckpt else ckpt
    preds = np.stack([m.predict(x_test) for m in committee], axis=1)
    ensemble_pred = preds.mean(axis=1)

    y_acc_pred = y_acc_pred_leave_out.copy()
    y_acc_pred[:, yi_idx] = ensemble_pred

    acc_curve = np.zeros(len(lambdas))
    cost_curve = np.zeros(len(lambdas))
    for i, lam in enumerate(lambdas):
        acc_curve[i], cost_curve[i] = score_auc(
            y_acc_test, y_cost_test, y_acc_pred, y_cost_test, lam
        )

    return acc_curve, cost_curve


directories = [
    "qbc_XGB_checkpoints_yi_42",
]

for d in directories:
    qbc_sizes = list(range(100, 11551, 100))

    qbc_curves = {}
    for n in qbc_sizes:
        result = load_and_eval(d, n)
        if result is not None:
            qbc_curves[n] = result

    qbc_chunks = [qbc_sizes[i:i+10] for i in range(0, len(qbc_sizes), 10)]

    markers = ['v', '^', '<', '>', '1', '2', '3', '4', '8', 's', 'p']
    colors_llms = ['b', 'g', 'r', 'c', 'm', 'y', 'k', 'brown', 'purple', 'gray', 'red']

    for chunk_sizes in qbc_chunks:
        plt.figure(figsize=(20, 8))
        plt.ylim(0.65, 0.825)
        plt.xlim(-0.01, 0.25)

        # baseline lines
        plt.plot(
            cost_xgb_baseline, acc_xgb_baseline, '-',
            color='green', label='XGBoost Baseline (all trained)', linewidth=2
        )
        plt.plot(
            cost_leave_out, acc_leave_out, '-',
            color='red', label='Leave Yi Out', linewidth=2
        )

        # QBC curves for this chunk only
        qbc_colors = plt.cm.tab10(np.linspace(0, 1, len(chunk_sizes)))
        for i, n in enumerate(chunk_sizes):
            if n in qbc_curves:
                acc_curve, cost_curve = qbc_curves[n]
                plt.plot(
                    cost_curve, acc_curve, '--',
                    color=qbc_colors[i],
                    label=f'QBC {n} samples',
                    linewidth=1.5
                )

        # individual LLM points
        for j, _ in enumerate(llms):
            avg_acc = np.mean(y_acc_test[:, j])
            avg_cost = np.mean(y_cost_test[:, j]) / np.max(y_cost_test)
            plt.scatter(
                avg_cost, avg_acc,
                marker=markers[j],
                color=colors_llms[j],
                s=80, alpha=0.7,
                label=llms_short[j]
            )

        start_n = chunk_sizes[0]
        end_n = chunk_sizes[-1]

        plt.xlabel('Cost', fontsize=12)
        plt.ylabel('Accuracy', fontsize=12)
        plt.title(
            f'QBC Active Learning Performance ({start_n}-{end_n} samples)',
            fontsize=14
        )
        plt.grid(True, alpha=0.3)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
        plt.tight_layout()
        plt.show()